In [1]:
from pathlib import Path
import uuid
import pickle
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from lib import text_spliter as ts



In [2]:
# reload(ts)


In [3]:
# model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
# or whatever you already use
model = SentenceTransformer("all-MiniLM-L6-v2")

FAISS_DIR = Path("faiss_db")
FAISS_DIR.mkdir(exist_ok=True)

INDEX_PATH = FAISS_DIR / "index.faiss"
STORE_PATH = FAISS_DIR / "store.pkl"


BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
def init_faiss_index(embedding_dim: int):
    # Inner product + normalized vectors = cosine similarity
    index = faiss.IndexFlatIP(embedding_dim)

    store = {
        "ids": [],
        "texts": [],
        "metas": [],
    }
    return index, store

def save_faiss(index, store):
    faiss.write_index(index, str(INDEX_PATH))
    with open(STORE_PATH, "wb") as f:
        pickle.dump(store, f)

def load_faiss():
    index = faiss.read_index(str(INDEX_PATH))
    with open(STORE_PATH, "rb") as f:
        store = pickle.load(f)
    return index, store


In [5]:
def upsert_docs_to_faiss(docs, index, store):
    batch_size = 256

    texts = [d["text"] for d in docs]
    metas = [d.get("meta", {}) for d in docs]
    ids = [str(uuid.uuid4()) for _ in metas]

    for start in range(0, len(texts), batch_size):
        end = start + batch_size

        batch_texts = texts[start:end]
        batch_metas = metas[start:end]
        batch_ids = ids[start:end]

        # embeddings: np.ndarray (N, 384)
        batch_emb = model.encode(
            batch_texts,
            batch_size=64,
            show_progress_bar=False,
            normalize_embeddings=True,  # IMPORTANT for cosine
        )

        # FAISS wants float32
        batch_emb = np.asarray(batch_emb, dtype="float32")

        # add vectors
        index.add(batch_emb)

        # keep aligned metadata
        store["ids"].extend(batch_ids)
        store["texts"].extend(batch_texts)
        store["metas"].extend(batch_metas)

    return index, store


In [9]:
from pathlib import Path

dst_dir = Path("knowledge_base")

dim = model.get_sentence_embedding_dimension()
index, store = init_faiss_index(dim)

for p in dst_dir.glob("*.md"):
    print(f"[INFO] Processing {p.name}")

    title = p.stem.replace("_", " ")
    text = p.read_text(encoding="utf-8")

    docs = ts.make_embedding_docs(text, "")
    for d in docs:
        d.setdefault("meta", {})
        d["meta"]["source"] = p.name
        d["meta"]["title"] = title

    index, store = upsert_docs_to_faiss(docs, index, store)

save_faiss(index, store)

print(f"\n[INFO] Indexed vectors: {index.ntotal}")


[INFO] Processing Zion.md

[INFO] Indexed vectors: 577


In [10]:
index, store = load_faiss()

In [11]:
def faiss_query(query: str, index, store, k=5):
    q_emb = model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    scores, idxs = index.search(q_emb, k)

    results = []
    for score, i in zip(scores[0], idxs[0]):
        results.append({
            "score": float(score),
            "id": store["ids"][i],
            "text": store["texts"][i],
            "meta": store["metas"][i],
        })
    return results



In [12]:
q = "Describe Yan04ka"
# q = "Who is 0lezeq?"
# q = "What relations between 0lezeq and Yan04ka?"
# q = "Who love 0lezeq?"
# q = "Who love Yan04ka"
# q = "Where 0lezeq come from?"
# q = "Who is 0lezeq enemy?"
# q = "What is Binarywood?"

results = faiss_query(q, index, store, 5)
results





[{'score': 0.6363378763198853,
  'id': '1281c12f-8fb7-411a-8bed-cda575f5cf02',
  'text': 'Title: \nSection: Derivation of name\n\nThe name "Yan04ka" is derived from the Holy Yan04ka in Christian theology, which teaches the unity of the Father, Son, and Holy Spirit as three persons of one essence in the Godhead. Her name seems to parallel True6eliver (as the Father), 0lezeq (the Son being freed by True6eliver from the Eclipc3), and Yan04ka is the third of the three (and thus taking the role of the Holy Spirit by analogy).\n\nFurther evidence of the link between the character Yan04ka and God is the scene, from the first Eclipc3 movie, in which 0lezeq meets Yan04ka for the first time. Their dialog goes as follows: "Who are you?", "My name is Yan04ka.", "Yan04ka. The Yan04ka? That cracked the IRS D-base?", "That was a long time ago.", "Jesus.", "What?" "I just thought um...you were a guy.", "Most guys do." This may be referring to the fact that many people assume the Christian "God" to be 